# Text classification with Transformer

## Dot-product and Multi-head attention
Dot-product and Multi-head attention from the paper "Attention is all you need" (2017). Implementation in modern Tensorflow 2 using the Keras API.

<img src="images/scaled-dot-product-attention-recap.png" alt="multi head self attention" width="800"/>

<img src="images/multi-head-attention.png" alt="multi head self attention" width="800"/>

### Scaled Dot-Product Attention

The scaled dot-product attention operates on three vectors: queries (Q), keys (K), and values (V). The process involves the following steps:

1. **Linear Transformation**: Each word's embedding is transformed into three vectors (Q, K, V) using different weight matrices $W^Q, W^K, W^V$. This is represented by matrix multiplication in the image (MatMul).

2. **Dot Product of Q and K**: The query matrix Q is multiplied by the transpose of the key matrix K to obtain a score matrix that represents the relationship between different words (Step 2 in the image). This score matrix indicates how much focus should be put on other parts of the input sentence when encoding a particular word.

3. **Scaling**: The scores are scaled down by the square root of the dimension of the key vectors $ \sqrt{d_k} $ (Step 3 in the image). This is done to prevent the softmax from having an extremely small gradient during training, which could happen if the dot products are very large.

4. **Softmax**: A softmax function is applied to the scaled scores to obtain the weights on the values (Step 4 in the image). The softmax function turns the scores into probabilities that sum to one.

5. **Output**: The softmax probabilities are then multiplied by the value matrix V (Step 5 in the image), and then the results of this multiplication are summed up to produce the final output for each word.

### Multi-Head Attention

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function, it linearly projects the queries, keys, and values multiple times with different, learned linear projections. This process involves:

1. **Linear Projections**: The queries, keys, and values are linearly projected $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions, respectively.

2. **Scaled Dot-Product Attention**: Each projected version of Q, K, and V computes the attention function in parallel, yielding $d_v$-dimensional output values: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

3. **Concatenation**: The $h$ parallel attention outputs are concatenated into a single matrix: $\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$, where:
   - $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$
   - $\text{Concat}(\text{head}_1, ..., \text{head}_h)$ represents the concatenation of the outputs from all the attention heads.

4. **Final Linear Transformation**: The concatenated matrix is again linearly transformed into the desired dimension.
   - $W^O$ is the final linear transformation matrix that projects the concatenated matrix back to the original embedding dimension.

The multi-head attention mechanism allows the model to capture information from different positions and representation subspaces, improving its ability to learn complex dependencies in the data.

Where $W_i^Q$, $W_i^K$, $W_i^V$, and $W^O$ are parameter matrices and $h$ is the number of heads.

In [1]:
import tensorflow as tf
import keras
from keras.layers import Dense, Input, Layer

### Dot product
Here we implement the dot product attention from scratch as a Keras layer.

In [2]:
class DotProductAttention(keras.layers.Layer):
    def __init__(self, use_scale=True, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.use_scale = use_scale

    def build(self, input_shape):
        """
        Create the state of the layer (weights).
        Args:
            input_shape: [(batch_size, n_vectors, d_model), (batch_size, n_vectors, d_model), (batch_size, n_vectors, d_model)]
        """
        query_shape = input_shape[0]
        if self.use_scale:
            dim_k = tf.cast(query_shape[-1], tf.float32)
            self.scale = 1 / tf.sqrt(dim_k)
        else:
            self.scale = None

    def call(self, input):
        """
        Defines the computation from inputs to outputs.
        Args:
            input: [query, key, value]
        """
        query, key, value = input
        score = tf.matmul(query, key, transpose_b=True)
        if self.scale is not None:
            score *= self.scale
        return tf.matmul(tf.nn.softmax(score), value)

### Multi-head attention
A basic implementation of the multi-head attention from scratch as a Keras layer.


In [3]:
class MultiHeadAttention(keras.layers.Layer):
    def __init__(self, h=8, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.h = h

    def build(self, input_shape):
        """
        Create the state of the layer (weights).
        Args:
            input_shape: [(batch_size, n_vectors, d_model), (batch_size, n_vectors, d_model), (batch_size, n_vectors, d_model)]
        """
        query_shape, key_shape, value_shape = input_shape
        d_model = query_shape[-1]

        # Note: units can be anything, but this is what the paper does
        units = d_model // self.h

        # Create h layers for each of query, key, value
        self.layersQ = []
        for _ in range(self.h):
            layer = Dense(units, activation=None, use_bias=False)
            layer.build(query_shape)
            self.layersQ.append(layer)

        self.layersK = []
        for _ in range(self.h):
            layer = Dense(units, activation=None, use_bias=False)
            layer.build(key_shape)
            self.layersK.append(layer)

        self.layersV = []
        for _ in range(self.h):
            layer = Dense(units, activation=None, use_bias=False)
            layer.build(value_shape)
            self.layersV.append(layer)

        # Create dot product attention layer
        self.attention = DotProductAttention(True)

        # Create output layer
        self.out = Dense(d_model, activation=None, use_bias=False)
        self.out.build((query_shape[0], query_shape[1], self.h * units))

    def call(self, input):
        """
        Defines the computation from inputs to outputs.
        Args:
            input: [query, key, value]
        """
        query, key, value = input

        q = [layer(query) for layer in self.layersQ]
        k = [layer(key) for layer in self.layersK]
        v = [layer(value) for layer in self.layersV]

        # Apply attention to each head
        head = [self.attention([q[i], k[i], v[i]]) for i in range(self.h)]

        # Concatenate heads
        out = self.out(tf.concat(head, -1))

        return out

Example use with some random data:

In [4]:
batch_size = 10     # Batch size for training.
n_vectors = 150     # Number of vectors in each sequence.
d_model = 512       # Dimensionality of the hidden representation.

# Randomly initialized query, key, value matrices
query = tf.random.uniform((batch_size, n_vectors, d_model), dtype=tf.float32)
key = tf.random.uniform((batch_size, n_vectors, d_model), dtype=tf.float32)
value = tf.random.uniform((batch_size, n_vectors, d_model), dtype=tf.float32)

# Test dot product attention
dp_layer = DotProductAttention(use_scale=True)
x = dp_layer([query, key, value])
print(f"Output from dot product attention: {x.shape}")

# Test multi-head attention
mh_layer = MultiHeadAttention(h=8)
x = mh_layer([query, key, value])
print(f"Output from multi-head attention: {x.shape}")

Output from dot product attention: (10, 150, 512)
Output from multi-head attention: (10, 150, 512)


## Developing Transformer Model From Scratch With TensorFlow and Keras:

### Importing the essential libraries:

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import LayerNormalization, Dropout, Layer, MultiHeadAttention
from tensorflow.keras.layers import Embedding, Input, GlobalAveragePooling1D, Dense
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential, Model
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

### Creating Transformer blocks and positional embedding:

In [6]:
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()

        # Multi-head attention
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        # Feed forward network
        self.ffn = Sequential(
            [Dense(ff_dim, activation="relu"), 
             Dense(embed_dim),]
        )

        # Normalization layers
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):

        # Multi-head attention
        attn_output = self.att(inputs, inputs)

        # Dropout and residual connection
        attn_output = self.dropout1(attn_output, training=training)

        # Add and normalize
        out1 = self.layernorm1(inputs + attn_output)

        # Feed forward network
        ffn_output = self.ffn(out1)

        # Dropout and residual connection
        ffn_output = self.dropout2(ffn_output, training=training)

        # Add and normalize
        out = self.layernorm2(out1 + ffn_output)
        
        return out

In [7]:
class TokenAndPositionEmbedding(Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

The importance of this layer is derived from its role in Transformer models. Transformers do not inherently understand the order of tokens in a sequence because they process the input data in parallel rather than sequentially. The position embeddings are a way to inject positional information into the model, allowing it to learn the significance of the order of tokens within the input sequences. This is critical for tasks such as language understanding, where the meaning of a sentence can change dramatically based on word order.

## IMDB Movie Review Sentiment Classification with Transformer:

### Preparing the data:

Here we will use the IMDB movie review dataset from the tensorflow datasets library for speed and convenience. To see the full pipeline for preparing the data, see the notebook on Text Classification with RNNs.

In [8]:
vocab_size = 10000  # Only consider the top 10k words
maxlen = 500  # Only consider the first 500 words of each movie review

(x_train, y_train), (x_val, y_val) = imdb.load_data(num_words=vocab_size)
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")

25000 Training sequences
25000 Validation sequences


In [9]:
y_val[:5]

array([0, 1, 1, 0, 1], dtype=int64)

In [10]:
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_val = tf.keras.preprocessing.sequence.pad_sequences(x_val, maxlen=maxlen)

### Developing the model:

The transformer encoder is a stack of N identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. To use the transformer block for text classification, we need to add a classification head on top of the output of the last layer. The classification head is a simple linear layer that maps the output of the last layer to the number of classes. We also add one or more dropout layers to prevent overfitting.

<img src="images/transformer-encoder-for-classification.png" alt="transformer encoder for classification" width="800"/>

In [11]:
embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer

# Inputs to the model
inputs = Input(shape=(maxlen,))

# Embedding layer
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)

# Transformer block
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

# A GlobalAveragePooling1D layer returns a fixed-length output vector for each example 
# by averaging over the sequence dimension. Here, it will average over the 500 
# token vectors.
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)

# Classification head and output layer
x = Dense(20, activation="relu")(x)
x = Dropout(0.1)(x)
outputs = Dense(2, activation="softmax")(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 500)]             0         
                                                                 
 token_and_position_embeddi  (None, 500, 32)           336000    
 ng (TokenAndPositionEmbedd                                      
 ing)                                                            
                                                                 
 transformer_block (Transfo  (None, 500, 32)           10656     
 rmerBlock)                                                      
                                                                 
 global_average_pooling1d (  (None, 32)                0         
 GlobalAveragePooling1D)                                         
                                                                 
 dropout_2 (Dropout)         (None, 32)                0     

### Compiling and fitting the model:

In [12]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = model.fit(x_train, y_train, 
                    batch_size=64, epochs=5, 
                    validation_data=(x_val, y_val)
                   )

Epoch 1/5
391/391 [==============================] - 232s 588ms/step - loss: 0.4578 - accuracy: 0.7601 - val_loss: 0.2792 - val_accuracy: 0.8831
Epoch 2/5
391/391 [==============================] - 233s 596ms/step - loss: 0.2171 - accuracy: 0.9180 - val_loss: 0.2710 - val_accuracy: 0.8902
Epoch 3/5
391/391 [==============================] - 234s 600ms/step - loss: 0.1720 - accuracy: 0.9384 - val_loss: 0.2907 - val_accuracy: 0.8816
Epoch 4/5
391/391 [==============================] - 246s 631ms/step - loss: 0.1297 - accuracy: 0.9545 - val_loss: 0.3352 - val_accuracy: 0.8745
Epoch 5/5
391/391 [==============================] - 231s 590ms/step - loss: 0.1048 - accuracy: 0.9649 - val_loss: 0.4198 - val_accuracy: 0.8679


### Evaluating the model:

In [13]:
results = model.evaluate(x_val, y_val, verbose=2)

for name, value in zip(model.metrics_names, results):
    print("%s: %.3f" % (name, value))

782/782 - 76s - loss: 0.4198 - accuracy: 0.8679 - 76s/epoch - 97ms/step
loss: 0.420
accuracy: 0.868
